# 🔬 Master Pipeline Integrado: Segmentação e Classificação A/V

Este notebook representa a versão final e unificada do pipeline. Em vez de declarar milhares de linhas de código (modelos, datasets e loops de treinamento) diretamente nas células, **este notebook consome a nova arquitetura Codebased (`src/`)** que acabamos de refatorar.

O objetivo é demonstrar como o pipeline pode ser executado de ponta a ponta (End-to-End) de forma limpa, científica e rastreável, utilizando os hiperparâmetros consolidados como os melhores entre todos os experimentos passados.

In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import cv2
import torch
from pathlib import Path

# Adicionar raiz do projeto ao path para importar a pasta src/
sys.path.append(os.path.abspath('..'))

from src.config.settings import SEGMENTATION_CONFIG, AV_CLASSIFICATION_CONFIG
from src.pipeline.integrated_pipeline import ScientificAVRPipeline

print("✅ Módulos do Codebase carregados com sucesso!")

## 1. Inicializando o Pipeline Científico
A configuração de parâmetros foi totalmente transposta para `src/config/settings.py`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Dispositivo configurado: {device}")

pipeline = ScientificAVRPipeline(device=device)

# Tenta carregar os pesoa caso existam (se já tiverem sido treinados)
model_loaded = pipeline.load_models()
if model_loaded:
    print("🧠 Modelos carregados e prontos para inferência!")
else:
    print("⚠️ Pesos de modelos não encontrados. É necessário executar o treinamento via `python main.py --train_seg --train_av`!")

## 2. Inferência em Imagem Isolada
Processo unificado: Extração de imagem -> Segmentação -> Classificação A/V.

In [ ]:
# Caminho de exemplo (modifique conforme necessário)
sample_image_path = "../data/DRIVE/test/images/01_test.tif"

if os.path.exists(sample_image_path):
    print(f"Processando imagem: {sample_image_path}...")
    
    # Segmentação Vascular
    seg_result = pipeline.segment_vessels(sample_image_path)
    vessel_mask = seg_result["mask"]
    print(f"✓ Vasos segmentados em {seg_result['inference_time_ms']:.2f} ms")
    
    # Classificação Artéria/Veia
    av_result = pipeline.classify_av(vessel_mask)
    av_pred = av_result["av_mask"]
    print(f"✓ Classificação concluída em {av_result['inference_time_ms']:.2f} ms")
    
    # Visualização
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    img_rgb = cv2.cvtColor(cv2.imread(sample_image_path), cv2.COLOR_BGR2RGB)
    axes[0].imshow(img_rgb)
    axes[0].set_title("Imagem Original")
    axes[0].axis("off")
    
    axes[1].imshow(vessel_mask, cmap='gray')
    axes[1].set_title("Vasos Segmentados (EnhancedUNet)")
    axes[1].axis("off")
    
    # Onde 1=Artéria (Vermelho), 2=Veia (Azul)
    colored_av = np.zeros((*av_pred.shape, 3), dtype=np.uint8)
    colored_av[av_pred == 1] = [255, 0, 0]   # Artérias
    colored_av[av_pred == 2] = [0, 0, 255]   # Veias
    
    axes[2].imshow(colored_av)
    axes[2].set_title("Classificação A/V (MultiDatasetAVNet)")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("As imagens de teste não estão disponíveis. Adicione imagens para verificar a inferência.")